In [ ]:
import os
with open("./job.list", "w") as f:
    for file in sorted(os.listdir("./inputs")):
        if file.endswith(".pdb"):
            filename = file[:-4]
        f.write( "/home/chuwang_pkuhpc/lustre1/jobs/cjj/install/rosetta.source.release-425/main/source/bin/relax.mpi.linuxgccrelease "
                 f"-s ./inputs/{file} "
                 "-relax:constrain_relax_to_start_coords "
                 "-ramp_constraints false "
                 "-relax:coord_constrain_sidechains "
                 "-nstruct 40 "
                 "-out:path:all ./outputs "
                 "-out:file:silent_struct_type binary "
                f"-out:file:silent {filename}.out.gz "
                f"-out:file:scorefile {filename}.sc "
                 "-ex1 "
                 "-ex2 "
                 "-ignore_zero_occupancy false "
                 "-use_input_sc "
                 "-flip_HNQ "
                 "-no_optH false "
                 "-score:weights beta_jan25 "
                f"-beta_jan25 > ./outputs/logs/{filename}_relaxed.log 2>&1\n" )

In [2]:
import os
with open("interface.list", "w") as f:
    for file in sorted(os.listdir("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/Dock/4_GLM/validation/generalization/rosetta/outputs")):
        if file.endswith(".pdb"):
            path = os.path.join("/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/Dock/4_GLM/validation/generalization/rosetta/outputs", file)
            f.write(f"InterfaceAnalyzer.mpi.linuxgccrelease -s {path} -out:file:score_only /home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/Dock/4_GLM/validation/generalization/rosetta/interface/packed_interface_score-betajan25.sc -beta_jan25 @/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/Dock/4_GLM/validation/interface/pack_input_options_nopackstat.txt > /home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/Dock/4_GLM/validation/generalization/rosetta/interface/{file}.log 2>&1\n")


In [4]:
import os

# 检查input中的每个文件是否都有40个输出结果
input_files = sorted([f for f in os.listdir("./inputs") if f.endswith(".pdb.gz")])
output_files = sorted([f for f in os.listdir("./outputs") if f.endswith(".sc")])
lost_num = 0

for input_file in input_files:
    base_name = input_file[:-7]  # 去掉 .pdb.gz
    matching_outputs = f'{base_name}.sc'
    if not os.path.exists(f"./outputs/{matching_outputs}"):
        print(f"文件 {input_file} 的输出结果缺失: 没有找到 {matching_outputs}")
        lost_num += 1
    else:
        with open(f"./outputs/{matching_outputs}", "r") as f:
            lines = f.readlines()
            if len(lines) != 42:
                print(f"文件 {input_file} 的输出结果数量不正确: 找到 {len(lines)} 行，预期 42 行")
                lost_num += 1

print(f"总共有 {lost_num} 个输入文件的输出结果缺失或数量不正确。")


文件 5e33_10_seed-42_sample-0_model.pdb.gz 的输出结果数量不正确: 找到 24 行，预期 42 行
文件 5e33_10_seed-42_sample-3_model.pdb.gz 的输出结果数量不正确: 找到 24 行，预期 42 行
文件 5e33_10_seed-44_sample-0_model.pdb.gz 的输出结果数量不正确: 找到 24 行，预期 42 行
文件 5e33_1_seed-43_sample-1_model.pdb.gz 的输出结果数量不正确: 找到 22 行，预期 42 行
文件 5e33_1_seed-43_sample-4_model.pdb.gz 的输出结果数量不正确: 找到 23 行，预期 42 行
文件 5e33_1_seed-44_sample-4_model.pdb.gz 的输出结果数量不正确: 找到 24 行，预期 42 行
文件 5e33_2_seed-42_sample-2_model.pdb.gz 的输出结果数量不正确: 找到 25 行，预期 42 行
文件 5e33_2_seed-43_sample-4_model.pdb.gz 的输出结果数量不正确: 找到 25 行，预期 42 行
文件 5e33_2_seed-44_sample-3_model.pdb.gz 的输出结果数量不正确: 找到 24 行，预期 42 行
文件 5e33_3_seed-42_sample-1_model.pdb.gz 的输出结果数量不正确: 找到 24 行，预期 42 行
文件 5e33_3_seed-43_sample-4_model.pdb.gz 的输出结果数量不正确: 找到 24 行，预期 42 行
文件 5e33_3_seed-44_sample-4_model.pdb.gz 的输出结果数量不正确: 找到 25 行，预期 42 行
文件 5e33_4_seed-42_sample-1_model.pdb.gz 的输出结果数量不正确: 找到 23 行，预期 42 行
文件 5e33_4_seed-43_sample-3_model.pdb.gz 的输出结果数量不正确: 找到 24 行，预期 42 行
文件 5e33_4_seed-44_sample-4_model.pdb.gz 的输出结果

In [2]:
# 提取每个sc文件中total_score最低的description，将对应的description组成一个list
import os
import pandas as pd 
descriptions = []
for file in sorted(os.listdir("./outputs/sc")):
    if file.endswith(".sc") and (not file.startswith("6g5g")) and (not file.startswith("5e33")) and (not file.startswith("6e5x")):
        df = pd.read_csv(f"./outputs/sc/{file}", skiprows=1, sep=r'\s+')
        min_score_row = df.loc[df['total_score'].idxmin()]
        descriptions.append(min_score_row['description'])
print(len(descriptions))


3210


In [9]:
with open("extract_pdb.sh", "w") as f:
    for description in descriptions:
        file = "_".join(description.split("_")[:-1])
        f.write(f"/home/chuwang_pkuhpc/lustre1/jobs/cjj/install/rosetta.source.release-425/main/source/bin/extract_pdbs.mpi.linuxgccrelease -in:file:silent /lustre1/chuwang_pkuhpc/jobs/cjj/Dpepdesign/rosetta/GLM/relax_af3_train/outputs/{file}.out.gz -in:file:tags \"{description}\" -out:path:all /lustre1/chuwang_pkuhpc/jobs/cjj/Dpepdesign/rosetta/GLM/relax_af3_train/lowest_outputs\n")

In [10]:
with open("extract_silent_add_header.sh", "w") as f:
    for description in descriptions:
        file = "_".join(description.split("_")[:-1])
        f.write(f"zcat /lustre1/chuwang_pkuhpc/jobs/cjj/Dpepdesign/rosetta/GLM/relax_af3_train/outputs/{file}.out.gz | awk 'NR<=3 || /{description}/' >> /lustre1/chuwang_pkuhpc/jobs/cjj/Dpepdesign/rosetta/GLM/relax_af3_train/lowest_outputs/silent/{description}.out\n")

In [4]:
with open("split_silent.sh", "w") as f:
    for description in descriptions:
        f.write(f"grep '{description}' ./outputs/all_except_6g5g_5e33_6e5x.out > ./outputs/silent_file/{description}.out\n")

In [ ]:
with open("./interface/jobs.list", "w") as f:
    for description in descriptions:
        f.write(f"InterfaceAnalyzer.mpi.linuxgccrelease -s /home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/Dock/4_GLM/2_combine_af3_train/rosetta/outputs/pdb/{description}.pdb -out:file:score_only af3_samples-interface-betajan25-nopackinput.sc -beta_jan25 @/home/junjiechen/1_work/250401-Dpepalign/Benchmark/Rosetta/interface/Dock/flags/pack_separate_options.txt > ./logs/{description}_pack_separated.log 2>&1\n")